In [1]:
from pathlib import Path
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


c:\Users\shravan\Documents\OMNI_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

FAISS_INDEX_PATH = PROCESSED_DIR / "faiss.index"
FAISS_META_PATH = PROCESSED_DIR / "faiss_chunks_meta.pkl"

SIM_THRESHOLD = 0.7
TOP_K = 8


In [3]:
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
print("FAISS vectors:", faiss_index.ntotal)

assert faiss_index.ntotal > 0


FAISS vectors: 23


In [4]:
with open(FAISS_META_PATH, "rb") as f:
    nodes = pickle.load(f)

print("Loaded nodes:", len(nodes))
assert len(nodes) == faiss_index.ntotal


Loaded nodes: 23


In [5]:
embed_model = SentenceTransformer("BAAI/bge-base-en")
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print("Embedding dim:", EMBED_DIM)
assert EMBED_DIM == faiss_index.d


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 459.08it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 768


In [6]:
def retrieve_chunks(query: str, k: int = TOP_K):
    query_vec = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        node = nodes[idx]
        results.append({
            "score": float(score),
            "text": node.text,
            "metadata": node.metadata
        })

    return results


In [7]:
def gated_retrieval(query: str):
    results = retrieve_chunks(query)

    confident = [
        r for r in results
        if r["score"] >= SIM_THRESHOLD
    ]

    return confident


In [8]:
def build_context(chunks):
    context_blocks = []

    for i, c in enumerate(chunks):
        block = f"""
[Evidence {i+1}]
Score: {c['score']}
Source: {c['metadata'].get('file_name')}

{c['text']}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)


In [9]:
def should_answer(chunks):
    if not chunks:
        return False, "No confident retrieval found."

    if max(c["score"] for c in chunks) < SIM_THRESHOLD:
        return False, "Retrieved evidence is too weak."

    return True, None


In [10]:
def agentic_rag(query: str):
    retrieved = gated_retrieval(query)

    ok, reason = should_answer(retrieved)
    if not ok:
        return {
            "answer": None,
            "reason": reason,
            "chunks": []
        }

    context = build_context(retrieved)

    return {
        "answer": "LLM_CAN_ANSWER",
        "context": context,
        "chunks": retrieved
    }


In [11]:
query = "How do I set up the OmniRAG environment?"

output = agentic_rag(query)

if output["answer"] is None:
    print("❌ Agent refused to answer")
    print("Reason:", output["reason"])
else:
    print("✅ Agent approved answer")
    print("\n--- CONTEXT SENT TO LLM ---\n")
    print(output["context"][:1500])


✅ Agent approved answer

--- CONTEXT SENT TO LLM ---

[Evidence 1]
Score: 0.7470496296882629
Source: ilide.info-python-interview-qa-datascience-genai-pr_205ecf461f656db3594cf7d5e9953c1d.pdf

What is Hugging Face Transformers library and how have you used it? Answer: It's a Python library to work with pretrained NLP models. I've used it to build
question-answering, text generation, and RAG-based systems. 17. How do you fine-tune a pre-trained language model using Python? Answer: Use Hugging Face Trainer API with a custom dataset and define training arguments,
tokenizer, and model. Call 'trainer.train()'. 18. What is tokenization in NLP and how is it implemented in Python? Answer: Tokenization splits text into tokens. In Transformers, use model-specific tokenizers like
'AutoTokenizer.from_pretrained(...)'. 19. Explain the concept of embeddings and how you generate them using Python. Answer: Embeddings are vector representations of text. Generate them using models like BERT
or Sentence Tr

In [12]:
bad_query = "Explain quantum gravity in OmniRAG"

output = agentic_rag(bad_query)

if output["answer"] is None:
    print("✅ Correctly refused hallucination")
    print("Reason:", output["reason"])
else:
    print("❌ This should NOT have passed")


❌ This should NOT have passed


In [13]:
def format_prompt(context: str, query: str):
    return f"""
You are a factual assistant.
Answer ONLY using the evidence below.
If the evidence is insufficient, say "I don't know".

{context}

Question:
{query}
""".strip()


In [14]:
from typing import TypedDict, List, Literal


In [15]:
class AgentState(TypedDict):
    query: str
    intent: Literal["factual", "multi_hop", "summary"]
    top_k: int
    retrieved_chunks: List[dict]
    answer: str
    llm_calls: int


In [16]:
def classify_intent_no_llm(query: str):
    q = query.lower()

    if "summarize" in q or "overview" in q:
        return "summary", 8

    if "difference" in q or "compare" in q or "how does" in q:
        return "multi_hop", 6

    return "factual", 3


In [17]:
def planner_node(state: AgentState) -> AgentState:
    intent, k = classify_intent_no_llm(state["query"])

    state["intent"] = intent
    state["top_k"] = k
    state["llm_calls"] += 1   # simulated LLM call

    return state


In [18]:
# ---- RETRIEVER STUBS (Phase 6, no DB dependency) ----

def faiss_search(query: str, k: int = 5):
    # simulate FAISS results
    return [
        {
            "text": f"[FAISS simulated chunk {i}]",
            "metadata": {"source": "faiss", "chunk_id": i},
            "score": 0.1 * i
        }
        for i in range(k)
    ]


def chroma_search(query: str, k: int = 5):
    # simulate Chroma results
    return [
        {
            "text": f"[Chroma simulated chunk {i}]",
            "metadata": {"source": "chroma", "chunk_id": i},
            "score": 0.1 * i
        }
        for i in range(k)
    ]


In [19]:
def retrieval_node(state: AgentState) -> AgentState:
    if state["intent"] == "summary":
        results = chroma_search(
            state["query"],
            k=state["top_k"]
        )
    else:
        results = faiss_search(
            state["query"],
            k=state["top_k"]
        )

    state["retrieved_chunks"] = results
    return state


In [20]:
def answer_node(state: AgentState) -> AgentState:
    state["llm_calls"] += 1   # 👈 WOULD have been an LLM call

    state["answer"] = (
        f"[SIMULATED ANSWER]\n"
        f"Intent: {state['intent']}\n"
        f"Used {len(state['retrieved_chunks'])} chunks"
    )

    return state


In [21]:
def run_agent(query: str) -> AgentState:
    state: AgentState = {
        "query": query,
        "intent": "",
        "retrieved_chunks": [],
        "answer": "",
        "llm_calls": 0
    }

    state = planner_node(state)
    state = retrieval_node(state)
    state = answer_node(state)

    return state


In [22]:
query = "How does agentic RAG differ from standard RAG?"

result = run_agent(query)

print("Answer:")
print(result["answer"])
print("\nLLM calls avoided:", result["llm_calls"])


Answer:
[SIMULATED ANSWER]
Intent: multi_hop
Used 6 chunks

LLM calls avoided: 2
